# 04 Model Evaluation on Validation Set

In [1]:
import subprocess
import sys
import os

# Specify the path to the folder containing your module
repo_root = '../'

src_path = os.path.join(repo_root, 'src')
# Add src_path to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [2]:
from custom_models import AnimalTemporalClassifier
from custom_models import AnimalClassifier
from custom_datasets import S3ImageWithTimeFeatureDataset
from unsupervised_evaluate import UnsupervisedEvaluator as UnsupervisedMetricsEvaluator
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
import pandas as pd
import time
from confidence_estimator import ConfidenceEstimator as ConfidenceEstimator


In [3]:
import boto3

session = boto3.Session(profile_name='jop-training-sso')

sts = session.client('sts')
identity = sts.get_caller_identity()
role = identity['Arn']
s3_client = session.client('s3')
display(role)


'arn:aws:sts::324183265896:assumed-role/AWSReservedSSO_aai-590-training-permission_a601dd5621a14868/geoffrey@aai-590'

In [4]:
MODELS_DICT = {
    0: {'name' : 'BuiltFromScratch', 'base' : 'None', 'use_custom_loss' : False},
    1: {'name' : 'Resnet18_ft', 'base' : 'AnimalClassifier', 'use_custom_loss' : False},
    2: {'name' : 'Resnet18_ft_withTemporalVector', 'base' : 'AnimalTemporalClassifier', 'use_custom_loss' : False},
    3: {'name' : 'Resnet18_ft_customLoss','base' : 'AnimalClassifier', 'use_custom_loss' : True},
    4: {'name' : 'Resnet18_ft_withTemporalVector_customLoss','base' : 'AnimalTemporalClassifier','use_custom_loss' : True}
}

## Step 0. Configure Input Parameters for Evaluation

In [25]:
# Configure/Derive parameters
NUM_CLASSES = 17

#=== Model Id Settings========
MODEL_NUMBER = 3
MODEL_NAME = MODELS_DICT[MODEL_NUMBER]['name']
MODEL_BASE = MODELS_DICT[MODEL_NUMBER]['base']
LOCAL_MODEL_DIR = f'./models/model{MODEL_NUMBER}'
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, 'model.pth')
if (MODEL_NUMBER == 1) | (MODEL_NUMBER == 3): USE_TEMPORAL_FEATURES = False
else: USE_TEMPORAL_FEATURES = True

# load label map json (ideally from same Model Name location)
LABEL_MAP_JSON = './data_split/train_val/label_mapping.json'
# load label map
label2idx = pd.read_json(LABEL_MAP_JSON, typ="series").to_dict()

# NEW DATASET CSV FILE
S3_NEW_DATA_CSV =  's3://aai-590-tmp2/data_split/train_val/val-meta.csv'

# Evaluation output
EVAL_DATA_DIR = os.path.join(repo_root, 'experiments2', 'evaluation',f'model{MODEL_NUMBER}')

# pred_probs output file
PRED_PROBS_FILE = os.path.join(EVAL_DATA_DIR,'pred_probs.csv')
PRED_LOGITS_FILE = os.path.join(EVAL_DATA_DIR, 'pred_logits.csv')




In [26]:
# create new data directory if it doesn't exist
if not os.path.exists(os.path.dirname(PRED_PROBS_FILE)):
    os.makedirs(os.path.dirname(PRED_PROBS_FILE))

In [27]:
MODEL_PATH

'./models/model3/model.pth'

In [28]:
USE_TEMPORAL_FEATURES

False

## Step 1. Load Most Recent Model and Label Mapping Used

In [29]:
# initialize custom model with same number of classes based on json file
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print(torch.backends.mps.is_available())  # Should be True
    print(torch.backends.mps.is_built())      # Should be True
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("DEVICE: ", device)

model = []
if MODEL_BASE == 'AnimalTemporalClassifier':
    # Load the model with temporal features
    model = AnimalTemporalClassifier(NUM_CLASSES).to(device)
else:
    # Load the model without temporal features
    model = AnimalClassifier(NUM_CLASSES).to(device)

# Load the weights from the .pth file (from BytesIO or file)
#model.load_state_dict(torch.load(BytesIO(pth_content), map_location=device))
#model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = model.to(device)
model.eval()

True
True
DEVICE:  mps


/opt/miniconda3/envs/pytorch_base/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/envs/pytorch_base/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


AnimalClassifier(
  (cnn): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, trac

## Step 2. Load new dataset into batches

In [30]:
# load dataset using custom class and session 

new_dataset = []
new_dataset = S3ImageWithTimeFeatureDataset(S3_NEW_DATA_CSV, session = session)
print(f"Number of images: {len(new_dataset)}")

new_dataset_loader = []
new_dataset_loader = DataLoader(new_dataset, batch_size=32, shuffle=False, num_workers=8)
print(f"Number of batches: {len(new_dataset_loader)}")

DEBUG input csv s3: aai-590-tmp2, data_split/train_val, val-meta.csv
DEBUG INFO: self.df.shape = (6833, 15)
DEBUG INFO: self.fs OK
DEBUG INFO: No Label Encoding needed for this dataset
DEBUG INFO: self.label2idx OK
DEBUG INFO: Dataset initialized with device mps
Number of images: 6833
Number of batches: 214


## Step 3. Perform Batch Inference & save prediction results to specific output location

In [14]:
import confidence_estimator as ConfidenceEstimator

In [ ]:
import importlib
importlib.reload(ConfidenceEstimator) 

In [31]:
#pred_labels = []
pred_logits = []
batch_id = 0

start_time = time.time()
model.eval()
model.to(device)

with torch.no_grad():
    #for image:
    #for batch_id in range(len(new_dataset_loader)):
    for images_batch, features_batch, scalars_batch in new_dataset_loader:
        batch_start_time = time.time()
        images, features = images_batch.to(device), features_batch.to(device)
        if USE_TEMPORAL_FEATURES:
            outputs = model(images, features)
        else:
            outputs = model(images) # logits/loss-specific output

        
        #_, predicted = torch.max(outputs, 1)
        
        #pred_labels_batch = [idx2label[int(idx)] for idx in predicted]

        #if (batch_id%20 == 0): 
        batch_end_time = time.time()
        elapsed_batch_time = batch_end_time - batch_start_time
        running_elapsed_time = batch_end_time - start_time
            #print(f"batch: {batch_id} of {len(new_dataset_loader)}")
            #print(f"outputs: {outputs}")
            #print(f"pred_label: {pred_labels_batch}")
            #print(f"Elapsed time: {elapsed_time:.4f} seconds")
            #start_time = time.time()
        print(f"--processed batch {batch_id}/{len(new_dataset_loader)} elapsed time: batch [{elapsed_batch_time} s]  total [{running_elapsed_time} s]", end='\r', flush=True)
        batch_id += 1
        #pred_labels.extend(pred_labels_batch)
        pred_logits.extend(outputs)

In [32]:
# store true labels
true_labels = new_dataset.df['label'].tolist()
# map true labels to indices
idx2label = {v: k for k, v in label2idx.items()}
# map true labels to indices
true_labels_idx = [label2idx[label] for label in true_labels]
# convert true labels to tensor
true_labels_tensor = torch.tensor(true_labels_idx, dtype=torch.long)

In [34]:
# transfer torch tensors from device to CPU, and convert to pred probabilities
pred_logits = torch.stack(pred_logits).cpu()
pred_probs = F.softmax(pred_logits, dim=1).numpy()

In [35]:
val_confidences = []
val_confidences = ConfidenceEstimator.ConfidenceEstimator(NUM_CLASSES)
val_confidences.calibrate(pred_logits, true_labels_tensor)

Calibrating confidence estimator using TvA histogram_binning...
DEBUG: histogram binning quantile
Calibration complete!


In [37]:
val_confidences.update_statistics(pred_logits)
val_conf_report = val_confidences.get_confidence_report()

In [38]:
val_conf_df = pd.DataFrame({
    'class' : val_conf_report['per_class'].keys()})

for i in range(0,NUM_CLASSES):
    val_conf_df.loc[i, 'label']= idx2label[i]
    val_conf_df.loc[i, 'count'] = val_conf_report['per_class'][f'class_{i}']['calibrated_confidence']['count'] 
    val_conf_df.loc[i, 'cal conf (mean)'] = val_conf_report['per_class'][f'class_{i}']['calibrated_confidence']['mean']
    val_conf_df.loc[i, 'cal conf (std)'] = val_conf_report['per_class'][f'class_{i}']['calibrated_confidence']['std'] 
    val_conf_df.loc[i, 'cal conf (p95)'] = val_conf_report['per_class'][f'class_{i}']['calibrated_confidence']['p95']
    val_conf_df.loc[i, 'orig conf (mean)'] = val_conf_report['per_class'][f'class_{i}']['original_confidence']['mean']
    val_conf_df.loc[i, 'orig conf (std)'] = val_conf_report['per_class'][f'class_{i}']['original_confidence']['std'] 
    val_conf_df.loc[i, 'orig conf (p95)'] = val_conf_report['per_class'][f'class_{i}']['original_confidence']['p95']
val_conf_df

,class,label,count,cal conf (mean),cal conf (std),cal conf (p95),orig conf (mean),orig conf (std),orig conf (p95)
0,class_0,car,209.0,0.831486,0.159749,0.985380,0.905246,0.144417,0.997013
1,class_1,coyote,884.0,0.856692,0.189158,0.997076,0.908466,0.164124,0.999945
2,class_2,deer,40.0,0.759924,0.224069,0.991373,0.824821,0.217445,0.999323
3,class_3,bobcat,383.0,0.843592,0.209189,0.999708,0.889491,0.185676,0.999967
4,class_4,dog,487.0,0.771729,0.225313,0.997076,0.835607,0.208205,0.999916
5,class_5,skunk,96.0,0.858463,0.167094,0.997076,0.917842,0.137918,0.999986
6,class_6,empty,14.0,0.384418,0.057677,0.446805,0.393074,0.110628,0.557970
7,class_7,cat,937.0,0.854723,0.194769,0.997076,0.905412,0.170070,0.999969
8,class_8,opossum,2053.0,0.947343,0.115319,1.000000,0.972683,0.090526,0.999961
9,class_9,squirrel,401.0,0.803566,0.205854,0.997076,0.871188,0.179174,0.999926


In [28]:
# save confidence measures to a file
confidence_global_df = pd.DataFrame(val_conf_report['global'])
confidence_global_df.to_csv(os.path.join(NEW_DATA_DIR, MODEL_NAME, 'confidence_global.csv'), index=False)
confidence_per_class_df = pd.DataFrame(val_conf_report['per_class']).T
confidence_per_class_df.to_csv(os.path.join(NEW_DATA_DIR, MODEL_NAME, 'confidence_per_class.csv'), index=True)

In [29]:
# transfer torch tensors from device to CPU, and convert to pred probabilities
pred_logits = torch.stack(pred_logits).cpu()
pred_probs = F.softmax(pred_logits, dim=1).numpy()

# Save pred probabilities to CSV
pred_logits_df = pd.DataFrame(pred_logits.numpy(), columns = label2idx.keys())
pred_logits_df.to_csv(PRED_LOGITS_FILE, index = False)
pred_probs_df = pd.DataFrame(pred_probs, columns = label2idx.keys())
pred_probs_df.to_csv(PRED_PROBS_FILE, index = False)

In [ ]:
# Calculate Global ECEs for uncalibrated and calibrated confidence scores
uncalibrated_ece = evaluator.expected_calibration_error(original_confidence, correctness, n_bins=n_bins)
calibrated_ece = evaluator.expected_calibration_error(calibrated_confidence, correctness, n_bins=n_bins)
print(f"Uncalibrated ECE: {uncalibrated_ece:.4f}")
print(f"Calibrated ECE: {calibrated_ece:.4f}")

In [40]:
pred_logits.shape

torch.Size([6833, 17])

In [44]:
_, pred_labels = torch.max(pred_logits, 1)

In [46]:
pred_labels.numpy()

array([8, 8, 8, ..., 8, 1, 1])

In [51]:
idx2label

{0: 'car',
 1: 'coyote',
 2: 'deer',
 3: 'bobcat',
 4: 'dog',
 5: 'skunk',
 6: 'empty',
 7: 'cat',
 8: 'opossum',
 9: 'squirrel',
 10: 'raccoon',
 11: 'rodent',
 12: 'rabbit',
 13: 'bird',
 14: 'badger',
 15: 'fox',
 16: 'lizard'}

In [52]:
pred_labels = [idx2label[pred_idx] for pred_idx in pred_labels.numpy()]

## Step 5. Evaluate on Ground Truth Labels (Validation Set)

In [53]:
# new dataset df
from sklearn.metrics import classification_report

# generate classification report
class_report = []
class_report = classification_report(
    y_true=true_labels,
    y_pred=pred_labels)

print(class_report)

              precision    recall  f1-score   support

      badger       0.00      0.00      0.00         4
        bird       0.89      0.62      0.73       429
      bobcat       0.79      0.85      0.82       355
         car       0.98      0.96      0.97       212
         cat       0.87      0.91      0.89       897
      coyote       0.85      0.87      0.86       862
        deer       0.97      0.23      0.37       171
         dog       0.83      0.77      0.80       524
       empty       0.07      0.17      0.10         6
         fox       0.00      0.00      0.00        13
      lizard       1.00      0.50      0.67         6
     opossum       0.97      0.96      0.97      2073
      rabbit       0.95      0.96      0.96       481
     raccoon       0.83      0.94      0.88       463
      rodent       0.00      0.00      0.00         1
       skunk       0.98      0.81      0.89       116
    squirrel       0.51      0.93      0.66       220

    accuracy              

/opt/miniconda3/envs/pytorch_base/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/pytorch_base/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/pytorch_base/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

## Step 6. Monitor/Trigger Retraining Pipeline